# V11 — MedMNIST + ResNet-50 (tab:consistency)
**Professor F**  
Goal: Verify BMIA holds on medical imaging (PathMNIST, 9 classes, RGB 28×28).  
Applies 3 synthetic corruptions (Gaussian noise, blur, contrast) to the test set.  
Uses ResNet-50 backbone (ImageNet pretrained → fine-tune head on PathMNIST).  
Output → `V11_MEDMNIST.json`

**Kaggle**: No extra datasets needed | GPU T4 x2 | Internet ON | ~20 min

In [ ]:
import subprocess
subprocess.run(['pip','install','medmnist','--quiet'],check=True)

import os,json,copy,gc,time
import numpy as np
import torch,torch.nn as nn,torch.optim as optim
import torchvision,torchvision.transforms as T
from torch.utils.data import DataLoader, TensorDataset
import medmnist
from medmnist import PathMNIST, INFO

torch.manual_seed(42); np.random.seed(42)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS=torch.cuda.device_count()
print(f'Device: {device}  GPUs: {N_GPUS}')
print(f'medmnist: {medmnist.__version__}')

NUM_CLASSES=9    # PathMNIST: 9 tissue types
IMG_SIZE=224     # resize 28→224 for ResNet-50
TTA_BATCH=64; EVAL_BATCH=256
LR=0.005; TAU=0.10
METHODS=['tent','sar','eata','bmia_fixed','bmia_adaptive','bmia_adaptive_v3']

# PathMNIST stats (RGB)
PM_MEAN=[0.7406,0.5331,0.7059]
PM_STD =[0.1230,0.1768,0.1022]

# Corruption functions (applied to normalised tensors)
def corrupt_gaussian(X,sev=3):
    stds=[0.02,0.04,0.06,0.08,0.10]
    return torch.clamp(X+torch.randn_like(X)*stds[sev-1],-3,3)

def corrupt_blur(X,sev=3):
    # Gaussian blur via average pool (approximate)
    ks=[3,3,5,5,7]; k=ks[sev-1]
    pad=k//2
    return nn.functional.avg_pool2d(X,k,stride=1,padding=pad)

def corrupt_contrast(X,sev=3):
    factors=[0.85,0.75,0.60,0.45,0.30]
    f=factors[sev-1]
    mean=X.mean(dim=(2,3),keepdim=True)
    return torch.clamp(mean+(X-mean)*f,-3,3)

CORRUPTIONS={
    'gaussian_noise': corrupt_gaussian,
    'blur':           corrupt_blur,
    'contrast':       corrupt_contrast,
}

print(f'NUM_CLASSES={NUM_CLASSES} (PathMNIST)  IMG_SIZE={IMG_SIZE}  LR={LR}')

In [ ]:
# ── Load PathMNIST + fine-tune ResNet-50 ─────────────────────────
tf_train=T.Compose([T.Resize(IMG_SIZE),T.RandomHorizontalFlip(),
                    T.ToTensor(),T.Normalize(PM_MEAN,PM_STD)])
tf_test =T.Compose([T.Resize(IMG_SIZE),T.ToTensor(),T.Normalize(PM_MEAN,PM_STD)])

ds_train=PathMNIST(split='train',transform=tf_train,download=True)
ds_test =PathMNIST(split='test', transform=tf_test, download=True)
dl_train=DataLoader(ds_train,batch_size=64,shuffle=True,num_workers=4,pin_memory=True)
print(f'Train: {len(ds_train)}  Test: {len(ds_test)}')

model=torchvision.models.resnet50(weights='IMAGENET1K_V1')
model.fc=nn.Linear(2048,NUM_CLASSES)
model=model.to(device)
print(f'ResNet-50 params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

# Phase 1: head only (3 ep)
for p in model.parameters(): p.requires_grad_(False)
for p in model.fc.parameters(): p.requires_grad_(True)
opt1=optim.Adam(model.fc.parameters(),lr=1e-3)
crit=nn.CrossEntropyLoss()
model.train(); t0=time.time()
for ep in range(3):
    for x,y in dl_train:
        x,y=x.to(device),y.squeeze(1).to(device)
        opt1.zero_grad(); crit(model(x),y).backward(); opt1.step()
    print(f'  Head ep={ep+1}/3  {(time.time()-t0)/60:.1f}min')

# Phase 2: all params, low LR (5 ep)
for p in model.parameters(): p.requires_grad_(True)
opt2=optim.SGD(model.parameters(),lr=0.01,momentum=0.9,weight_decay=1e-4)
sch=optim.lr_scheduler.CosineAnnealingLR(opt2,T_max=5)
for ep in range(5):
    for x,y in dl_train:
        x,y=x.to(device),y.squeeze(1).to(device)
        opt2.zero_grad(); crit(model(x),y).backward(); opt2.step()
    sch.step()
    print(f'  Full ep={ep+1}/5  {(time.time()-t0)/60:.1f}min')

model.eval(); SOURCE_STATE=copy.deepcopy(model.state_dict())
del opt1,opt2,dl_train,ds_train; gc.collect(); torch.cuda.empty_cache()
print(f'Done {(time.time()-t0)/60:.1f}min')

In [ ]:
# ── Pre-load test set to GPU tensor ──────────────────────────────
dl_test=DataLoader(ds_test,batch_size=256,shuffle=False,num_workers=4)
all_x,all_y=[],[]
for x,y in dl_test: all_x.append(x); all_y.append(y.squeeze(1))
X_clean=torch.cat(all_x).to(device)
Y_test =torch.cat(all_y).to(device)
print(f'Test set: {X_clean.shape}  classes: {Y_test.unique().numel()}')

# Source accuracy
model.eval()
with torch.no_grad():
    preds=torch.cat([model(X_clean[i:i+EVAL_BATCH]).argmax(1)
                     for i in range(0,X_clean.size(0),EVAL_BATCH)])
clean_acc=(preds==Y_test).float().mean().item()
print(f'Source acc (clean): {clean_acc*100:.1f}%')
assert clean_acc>0.5,'Training may have failed — clean acc too low'

# ── TTA helpers ──────────────────────────────────────────────────
def freeze_bn(mdl):
    ids={id(p) for m in mdl.modules() if isinstance(m,(nn.BatchNorm2d,)) for p in m.parameters()}
    for p in mdl.parameters(): p.requires_grad_(id(p) in ids)

def get_acc(mdl,X,y):
    mdl.eval(); preds=[]
    with torch.no_grad():
        for i in range(0,X.size(0),EVAL_BATCH): preds.append(mdl(X[i:i+EVAL_BATCH]).argmax(1))
    p=torch.cat(preds)
    nc=(torch.bincount(p,minlength=NUM_CLASSES)>0).sum().item()
    dom=torch.bincount(p,minlength=NUM_CLASSES).float().max().item()/p.size(0)
    return round((p==y[:len(p)]).float().mean().item(),4),round(dom,4),nc

def run_method(X,y,method):
    model.load_state_dict(copy.deepcopy(SOURCE_STATE))
    model.train(); freeze_bn(model)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=optim.SGD(params,lr=LR,momentum=0.9)
    init={pn:p.data.clone() for pn,p in model.named_parameters() if p.requires_grad}
    lam=({'bmia_fixed':1.0,'bmia_adaptive':0.5,'bmia_adaptive_v3':0.1}).get(method,0.5)
    is_fixed=(method=='bmia_fixed'); use_v3=(method=='bmia_adaptive_v3')
    step=TTA_BATCH*2 if use_v3 else TTA_BATCH; fisher=None

    if method=='eata':
        fisher={pn:torch.zeros_like(p) for pn,p in model.named_parameters() if p.requires_grad}
        for j in range(0,min(128,X.size(0)),TTA_BATCH):
            xb=X[j:j+TTA_BATCH]
            if xb.size(0)<2: continue
            model.zero_grad(); pr=torch.softmax(model(xb),1)
            (-(pr*torch.log(pr+1e-8)).sum(1).mean()).backward()
            for pn,p in model.named_parameters():
                if pn in fisher and p.grad is not None: fisher[pn]+=p.grad.data**2*xb.size(0)
        for pn in fisher: fisher[pn]/=128
        model.load_state_dict(copy.deepcopy(SOURCE_STATE)); model.train(); freeze_bn(model)
        params=[p for p in model.parameters() if p.requires_grad]
        opt=optim.SGD(params,lr=LR,momentum=0.9)

    for i in range(0,X.size(0),step):
        if use_v3:
            chunks=[X[i+s*TTA_BATCH:i+(s+1)*TTA_BATCH] for s in range(2)]
            chunks=[c for c in chunks if c.size(0)>=4]
            if not chunks: continue
            xb=torch.cat(chunks)
        else:
            xb=X[i:i+TTA_BATCH]
            if xb.size(0)<4: continue
        opt.zero_grad()
        pr=torch.softmax(model(xb),1); ent=-(pr*torch.log(pr+1e-8)).sum(1)
        if method=='tent': loss=ent.mean()
        elif method=='sar':
            mask=ent<0.4*np.log(NUM_CLASSES)
            if mask.sum()<2: continue
            loss=ent[mask].mean()
        elif method=='eata':
            mask=ent<0.4*np.log(NUM_CLASSES)
            if mask.sum()<2: continue
            fl=sum((fisher[pn]*(p-init[pn])**2).sum() for pn,p in model.named_parameters() if pn in fisher)
            loss=ent[mask].mean()+2000*fl
        else:
            mg=pr.mean(0); hy=-(mg*torch.log(mg+1e-8)).sum()
            anc=sum(((p-init[pn])**2).sum() for pn,p in model.named_parameters() if pn in init)
            loss=ent.mean()-hy+lam*anc
        loss.backward(); opt.step()
        if not is_fixed and method in('bmia_adaptive','bmia_adaptive_v3'):
            with torch.no_grad():
                dom=(torch.bincount(pr.argmax(1),minlength=NUM_CLASSES).float().max()/xb.size(0)).item()
            lam=min(10.,lam*1.1) if dom>TAU else max(0.01,lam*0.95)
    return get_acc(model,X,y)

print('Helpers ready.')

In [ ]:
# ── Run 3 corruptions × 6 methods ────────────────────────────────
results=[]; t0=time.time()
print(f'  {"Corruption":<18} {"Method":<22} Acc    Dom    Cls')
for corr_name,corr_fn in CORRUPTIONS.items():
    X_corr=corr_fn(X_clean.clone(),sev=3)   # severity 3 of 5
    for m in METHODS:
        acc,dom,nc=run_method(X_corr,Y_test,m)
        col=(dom>0.15 and nc<9) or nc<5   # 9-class collapse threshold
        results.append({'corruption':corr_name,'method':m,'acc':acc,
                        'dom_pct':dom,'n_classes':nc,'collapsed':col})
        print(f'  {corr_name:<18} {m:<22} {acc*100:.1f}%  {dom*100:.1f}%  {nc}'
              f'  {"[COL]" if col else ""}')
        gc.collect(); torch.cuda.empty_cache()
    del X_corr; gc.collect(); torch.cuda.empty_cache()

print(f'\nDone {(time.time()-t0)/60:.1f}min')
print('\nAvg per method:')
for m in METHODS:
    avg=np.mean([r['acc'] for r in results if r['method']==m])*100
    print(f'  {m:<22} {avg:.1f}%')

with open('V11_MEDMNIST.json','w') as f:
    json.dump({'experiment':'V11_MEDMNIST','backbone':'ResNet-50',
               'dataset':'PathMNIST','num_classes':NUM_CLASSES,
               'clean_acc':clean_acc,'lr':LR,'severity':3,
               'corruptions':list(CORRUPTIONS.keys()),
               'results':results},f,indent=2)
print('Saved → V11_MEDMNIST.json')
with open('V11_MEDMNIST.json') as f: print(f.read())